In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv(r"/content/qoute_dataset.csv")

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [ ]:
quotes=df['quote']

In [ ]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
quotes=quotes.str.lower()

In [ ]:
import string
translator=str.maketrans('','',string.punctuation)
quotes=quotes.apply(lambda x:x.translate(translator))

In [ ]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size=10000
tok=Tokenizer(num_words=vocab_size,)
tok.fit_on_texts(quotes)


In [ ]:
word_index=tok.word_index

In [ ]:
print(len(word_index))

8978


In [ ]:
list(word_index.items())[:10]

[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [ ]:
sequence=tok.texts_to_sequences(quotes)

In [ ]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [ ]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [ ]:
X=[]
y=[]
for seq in sequence:
    for i in range(1,len(seq)):
        input_seq=seq[:i]
        output_seq=seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [ ]:
len(X),len(y)

(85271, 85271)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
max_len=max(len(x)for x in X)

In [ ]:
max_len

745

In [ ]:
X_padded=pad_sequences(X,maxlen=max_len,padding='pre')

In [ ]:
y=np.array(y)

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot=to_categorical(y,num_classes=vocab_size)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN, Dropout, Bidirectional

In [ ]:
embed_dim=50
rnn_unit=128

In [ ]:
rnn_model=Sequential()
rnn_model.add(Embedding(input_dim=vocab_size,
    output_dim=embed_dim,
    input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_unit))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim))
lstm_model.add(Bidirectional(LSTM(units=rnn_unit, return_sequences=True)))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(units=rnn_unit))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [ ]:
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epoch=100
batch_size=128


In [ ]:
# history_lstm = lstm_model.fit(X_padded, y_one_hot, epochs=100, batch_size=batch_size, validation_split=0.1)

In [ ]:
lstm_model.save('lstm_model.h5')

In [ ]:
from tensorflow.keras.models import load_model
lstm_model=load_model('/content/lstm_model.h5')

In [ ]:
index_to_word={}
for word,index in word_index.items():
  index_to_word[index]=word

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,tokenizer,text,max_len):
    text=text.lower()
    # The method is texts_to_sequences (plural)
    seq=tokenizer.texts_to_sequences([text])
    # seq is already a list containing one sequence
    seq=pad_sequences(seq,maxlen=max_len,padding='pre')

    pred=model.predict(seq,verbose=0)
    pred_index=np.argmax(pred)
    # Handle KeyError if pred_index is not in index_to_word
    predicted_word=index_to_word.get(pred_index, "") # Return empty string if key not found
    return predicted_word

In [ ]:
seed='what are you worrying'
nextword=predictor(lstm_model,tok,seed,max_len)
print(nextword)

seenbut


In [ ]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word=predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
        break
    seed_text+=" "+next_word
  return seed_text

In [ ]:
seed="where are "
generated_result = generate_text(lstm_model, tok, seed, max_len, 6)
print(generated_result)

where are  plant seenbut appalling appalling percy


In [ ]:
import pickle
with open('tokenizer.pkl','wb') as f:
    pickle.dump(tok,f)

In [ ]:
with open('max_len.pkl','wb') as f:
    pickle.dump(max_len,f)
